# LeetCode 1014 — Best Sightseeing Pair bằng Python

Link đề: https://leetcode.com/problems/best-sightseeing-pair/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def maxScoreSightseeingPair(self, values: List[int]) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất**. Điểm cốt lõi: điểm của cặp `(i, j)`, `i < j`, là `values[i] + values[j] + i − j = (values[i] + i) + (values[j] − j)` — **tách thành hai phần độc lập**; với mỗi `j` ta chỉ cần giá trị lớn nhất của `values[i] + i` trong các `i < j` ⇒ một lượt `O(n)`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chọn hai điểm tham quan `i < j` tối đa hoá `values[i] + values[j] + i − j`.
2. Nắm công cụ: **tách biến** (`A[i] = values[i] + i`, `B[j] = values[j] − j`), duy trì **cực đại tiền tố**, quét từ phải sang trái, chia để trị.
3. Đi từ **thử mọi cặp (`O(n²)`)** → **một lượt với cực đại tiền tố của `values[i] + i`** → **hai mảng cực đại tiền tố** → **chia để trị (qua điểm giữa)** → **quét ngược với cực đại hậu tố của `values[j] − j`**.
4. Hiểu mấu chốt: `score(i, j) = A[i] + B[j]` với `A[i] = values[i] + i`, `B[j] = values[j] − j`; với `j` cố định, `max_i<j score = (max_{i<j} A[i]) + B[j]`. Điều kiện `i < j` chỉ cần xử lý bằng thứ tự cập nhật.
5. Tránh lỗi kinh điển: cập nhật cực đại **trước** khi tính điểm (cho phép `i = j`, điểm `2·values[j]`), nhầm dấu khoảng cách (`i + j` hay `j − i`), chỉ xét cặp kề nhau, chọn hai giá trị lớn nhất rồi trừ khoảng cách (tham lam sai), quên giảm theo khoảng cách ở biến thể "phân rã".

## 2. Đề bài & ví dụ

Cho mảng `values`, `values[i]` là giá trị điểm tham quan thứ `i`. Khoảng cách giữa điểm `i` và `j` là `j − i`.

Điểm của cặp `(i, j)` với `i < j` là `values[i] + values[j] + i − j`: tổng giá trị trừ khoảng cách. Trả về **điểm lớn nhất** của một cặp.

**Ví dụ 1:** `values = [8,1,5,2,6]` → `11` (`i = 0`, `j = 2`: `8 + 5 + 0 − 2`).

**Ví dụ 2:** `values = [1,2]` → `2`.

**Ràng buộc:** `2 <= values.length <= 5·10^4`; `1 <= values[i] <= 1000`.

In [1]:
import itertools
import random
import time
from collections import deque
from itertools import accumulate
from typing import List

examples = [([8, 1, 5, 2, 6], 11), ([1, 2], 2), ([5, 5, 5], 9), ([10, 1, 1, 10], 17)]
for v, want in examples:
    print(f"values = {v} -> {want}")

values = [8, 1, 5, 2, 6] -> 11
values = [1, 2] -> 2
values = [5, 5, 5] -> 9
values = [10, 1, 1, 10] -> 17


## 3. Kiến thức nền

### 3.1 Tách biến

`values[i] + values[j] + i − j = (values[i] + i) + (values[j] − j)`. Định nghĩa `A[i] = values[i] + i` và `B[j] = values[j] − j`. Khi đó `score(i, j) = A[i] + B[j]` — một vế chỉ phụ thuộc `i`, vế kia chỉ phụ thuộc `j`.

In [2]:
v = [8, 1, 5, 2, 6]
A = [x + i for i, x in enumerate(v)]
B = [x - j for j, x in enumerate(v)]
print("A =", A, "| B =", B)
print("score(0, 2) =", A[0] + B[2], "= 8 + 5 + 0 - 2 =", 8 + 5 + 0 - 2)

A = [8, 2, 7, 5, 10] | B = [8, 0, 3, -1, 2]
score(0, 2) = 11 = 8 + 5 + 0 - 2 = 11


### 3.2 Cực đại tiền tố

Với `j` cố định, `max_{i<j} (A[i] + B[j]) = B[j] + max_{i<j} A[i]`. Giữ biến `best_left = max A[0..j−1]` khi duyệt `j`. Hàm `itertools.accumulate(A, max)` cho mọi cực đại tiền tố cùng lúc.

In [3]:
prefix_max = list(accumulate(A, max))
print("cực đại tiền tố của A:", prefix_max)
print("điểm tốt nhất kết thúc tại j:", [prefix_max[j - 1] + B[j] for j in range(1, len(v))])

cực đại tiền tố của A: [8, 8, 8, 8, 10]
điểm tốt nhất kết thúc tại j: [8, 11, 7, 10]


### 3.3 Thứ tự cập nhật

`i < j` được bảo đảm bằng cách **tính điểm trước, cập nhật `best_left` sau** (hoặc dùng `prefix_max[j − 1]`). Nếu cập nhật trước, `best_left` đã chứa `A[j]` và ta tính cả "cặp" `(j, j)` với điểm `A[j] + B[j] = 2·values[j]` — không hợp lệ.

## 4. Ý tưởng nền tảng / trực giác

`values = [8, 1, 5, 2, 6]`: `A = [8, 2, 7, 5, 10]`, `B = [8, 0, 3, −1, 2]`.

| `j` | `B[j]` | `best_left = max A[0..j−1]` | điểm tốt nhất với `j` |
|---|---|---|---|
| 1 | 0 | 8 | 8 |
| 2 | 3 | 8 | **11** |
| 3 | −1 | 8 | 7 |
| 4 | 2 | 8 | 10 |

Cực đại `11` tại `j = 2` (với `i = 0`) ✓. Một lượt, `O(n)` — không cần so mọi cặp.

## 5. Brute force — thử mọi cặp

**Ý tưởng:** hai vòng lặp `i < j`, tính `values[i] + values[j] + i − j`, lấy lớn nhất. `O(n²)` — với `n = 5·10^4` là `1.25·10^9` cặp, quá chậm; dùng làm **oracle**.

In [4]:
def sight_brute(values):
    n = len(values)
    best = None
    for i in range(n):
        for j in range(i + 1, n):
            s = values[i] + values[j] + i - j
            if best is None or s > best:
                best = s
    return best


for v, want in examples:
    assert sight_brute(v) == want
print([sight_brute(v) for v, _ in examples])

[11, 2, 9, 17]


In [5]:
# Code nộp LeetCode cho Brute force (O(n^2): quá chậm với n = 5·10^4)
class SolutionBrute:
    def maxScoreSightseeingPair(self, values: List[int]) -> int:
        best = float("-inf")
        for i in range(len(values)):
            for j in range(i + 1, len(values)):
                best = max(best, values[i] + values[j] + i - j)
        return best

**Độ phức tạp:** `O(n²)` thời gian, `O(1)` bộ nhớ.

**Nút thắt:** với mỗi `j`, ta duyệt lại mọi `i < j` để tìm `values[i] + i` lớn nhất. Giá trị đó có thể được **duy trì** khi `j` tăng.

## 6. Bước đột phá

`score = (values[i] + i) + (values[j] − j)`. Duy trì `best_left = max_{i<j}(values[i] + i)`; với mỗi `j` cập nhật `ans = max(ans, best_left + values[j] − j)`, rồi `best_left = max(best_left, values[j] + j)`. Một lượt `O(n)`, `O(1)` bộ nhớ.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** một lượt với `best_left` (đáp án chính), **Cách 3** hai mảng cực đại tiền tố (`accumulate`), **Cách 4** chia để trị, **Cách 5** quét ngược với cực đại hậu tố của `values[j] − j`.

### Cách 2 — Một lượt với `best_left`

**Giải thích từng đoạn:**

1. `best_left = values[0] + 0`, `ans = −∞`.
2. Với `j = 1..n−1`: `ans = max(ans, best_left + values[j] − j)`; sau đó `best_left = max(best_left, values[j] + j)`.

In [6]:
def sight_one_pass(values):
    best_left = values[0] + 0                         # max(values[i] + i) trên các i < j
    ans = float("-inf")
    for j in range(1, len(values)):
        ans = max(ans, best_left + values[j] - j)     # dùng best_left TRƯỚC khi cập nhật (đảm bảo i < j)
        best_left = max(best_left, values[j] + j)
    return ans


for v, want in examples:
    assert sight_one_pass(v) == want
print([sight_one_pass(v) for v, _ in examples])

[11, 2, 9, 17]


In [7]:
# Code nộp LeetCode cho Cách 2
class SolutionOnePass:
    def maxScoreSightseeingPair(self, values: List[int]) -> int:
        best_left = values[0]
        ans = float("-inf")
        for j in range(1, len(values)):
            ans = max(ans, best_left + values[j] - j)
            best_left = max(best_left, values[j] + j)
        return ans

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

### Cách 3 — Hai mảng cực đại tiền tố

**Ý tưởng:** `pm = list(accumulate(A, max))` với `A[i] = values[i] + i`. Với mỗi `j ≥ 1`: `pm[j − 1] + values[j] − j`; lấy `max`.

In [8]:
def sight_prefix_arrays(values):
    A = [x + i for i, x in enumerate(values)]
    pm = list(accumulate(A, max))                     # pm[k] = max(A[0..k])
    return max(pm[j - 1] + values[j] - j for j in range(1, len(values)))


for v, want in examples:
    assert sight_prefix_arrays(v) == want
print([sight_prefix_arrays(v) for v, _ in examples])

[11, 2, 9, 17]


In [9]:
# Code nộp LeetCode cho Cách 3
from itertools import accumulate


class SolutionPrefixArrays:
    def maxScoreSightseeingPair(self, values: List[int]) -> int:
        pm = list(accumulate((x + i for i, x in enumerate(values)), max))
        return max(pm[j - 1] + values[j] - j for j in range(1, len(values)))

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

### Cách 4 — Chia để trị

**Ý tưởng:** `solve(l, r)` trả về `(best, maxA, maxB)` trên đoạn `[l, r]`: `best` = điểm tốt nhất trong đoạn, `maxA = max(values[i] + i)`, `maxB = max(values[j] − j)`. Ghép hai nửa: điểm tốt nhất qua điểm giữa = `maxA(nửa trái) + maxB(nửa phải)` (mọi `i` ở nửa trái nhỏ hơn mọi `j` ở nửa phải).

In [10]:
def sight_divide(values):
    def solve(l, r):
        if l == r:
            return (float("-inf"), values[l] + l, values[l] - l)
        m = (l + r) // 2
        bl, al, bbl = solve(l, m)
        br, ar, bbr = solve(m + 1, r)
        cross = al + bbr                               # i ở nửa trái, j ở nửa phải
        return (max(bl, br, cross), max(al, ar), max(bbl, bbr))

    return solve(0, len(values) - 1)[0]


for v, want in examples:
    assert sight_divide(v) == want
print([sight_divide(v) for v, _ in examples])

[11, 2, 9, 17]


In [11]:
# Code nộp LeetCode cho Cách 4
class SolutionDivide:
    def maxScoreSightseeingPair(self, values: List[int]) -> int:
        def solve(l, r):
            if l == r:
                return (float("-inf"), values[l] + l, values[l] - l)
            m = (l + r) // 2
            bl, al, bbl = solve(l, m)
            br, ar, bbr = solve(m + 1, r)
            return (max(bl, br, al + bbr), max(al, ar), max(bbl, bbr))

        return solve(0, len(values) - 1)[0]

**Độ phức tạp:** `T(n) = 2T(n/2) + O(1)` ⇒ `O(n)` thời gian, `O(log n)` ngăn xếp.

### Cách 5 — Quét ngược với cực đại hậu tố

**Ý tưởng:** đối xứng: duyệt `i` từ phải sang trái, giữ `best_right = max_{j>i}(values[j] − j)`; điểm với `i` là `values[i] + i + best_right`.

In [12]:
def sight_scan_right(values):
    n = len(values)
    best_right = values[n - 1] - (n - 1)               # max(values[j] - j) trên các j > i
    ans = float("-inf")
    for i in range(n - 2, -1, -1):
        ans = max(ans, values[i] + i + best_right)     # dùng best_right TRƯỚC khi cập nhật
        best_right = max(best_right, values[i] - i)
    return ans


for v, want in examples:
    assert sight_scan_right(v) == want
print([sight_scan_right(v) for v, _ in examples])

[11, 2, 9, 17]


In [13]:
# Code nộp LeetCode cho Cách 5
class SolutionScanRight:
    def maxScoreSightseeingPair(self, values: List[int]) -> int:
        n = len(values)
        best_right = values[-1] - (n - 1)
        ans = float("-inf")
        for i in range(n - 2, -1, -1):
            ans = max(ans, values[i] + i + best_right)
            best_right = max(best_right, values[i] - i)
        return ans

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

## 8. TRACE từng bước

**Trace Cách 2** với `values = [8, 1, 5, 2, 6]`:

| `j` | `values[j]` | `best_left` (trước) | điểm `best_left + values[j] − j` | `ans` | `best_left` (sau) |
|---|---|---|---|---|---|
| (khởi tạo) | — | 8 | — | −∞ | 8 |
| 1 | 1 | 8 | `8 + 1 − 1 = 8` | 8 | `max(8, 2) = 8` |
| 2 | 5 | 8 | `8 + 5 − 2 = 11` | 11 | `max(8, 7) = 8` |
| 3 | 2 | 8 | `8 + 2 − 3 = 7` | 11 | `max(8, 5) = 8` |
| 4 | 6 | 8 | `8 + 6 − 4 = 10` | 11 | `max(8, 10) = 10` |

Đáp án `11` ✓. Hàm trace in từng bước (kèm `i` tốt nhất):

In [14]:
def trace_one_pass(values):
    best_left, best_i = values[0], 0
    ans, ans_pair = float("-inf"), None
    print("values =", values)
    for j in range(1, len(values)):
        score = best_left + values[j] - j
        if score > ans:
            ans, ans_pair = score, (best_i, j)
        print(f"  j={j}: best_left={best_left} (i={best_i}), điểm = {best_left} + {values[j]} - {j} = {score}, ans={ans}")
        if values[j] + j > best_left:
            best_left, best_i = values[j] + j, j
    print("=>", ans, "tại cặp", ans_pair)
    return ans, ans_pair


ans, pair = trace_one_pass([8, 1, 5, 2, 6])
assert ans == 11 and pair == (0, 2)

values = [8, 1, 5, 2, 6]
  j=1: best_left=8 (i=0), điểm = 8 + 1 - 1 = 8, ans=8
  j=2: best_left=8 (i=0), điểm = 8 + 5 - 2 = 11, ans=11
  j=3: best_left=8 (i=0), điểm = 8 + 2 - 3 = 7, ans=11
  j=4: best_left=8 (i=0), điểm = 8 + 6 - 4 = 10, ans=11
=> 11 tại cặp (0, 2)


**Trace lỗi thứ tự cập nhật:** nếu cập nhật `best_left` trước khi tính điểm, ở `j = 4` ta có `best_left = 10` và điểm `10 + 6 − 4 = 12`: tương ứng cặp `(4, 4)` — không hợp lệ (đáp án thành `12` thay vì `11`).

In [15]:
def wrong_order(values):
    best_left, ans = values[0], float("-inf")
    for j in range(1, len(values)):
        best_left = max(best_left, values[j] + j)          # BUG: cập nhật trước
        ans = max(ans, best_left + values[j] - j)
    return ans


print(wrong_order([8, 1, 5, 2, 6]), "!=", sight_one_pass([8, 1, 5, 2, 6]))
assert wrong_order([8, 1, 5, 2, 6]) == 12

12 != 11


## 9. Code nộp LeetCode

Chọn **Cách 2** (một lượt, `O(1)` bộ nhớ). Giữ nguyên `class Solution`, `maxScoreSightseeingPair(self, values)`:

In [16]:
class Solution:
    def maxScoreSightseeingPair(self, values: List[int]) -> int:
        best_left = values[0]                         # max(values[i] + i) trên các i < j đã duyệt
        ans = float("-inf")
        for j in range(1, len(values)):
            ans = max(ans, best_left + values[j] - j)  # tính điểm TRƯỚC khi đưa j vào best_left (i < j)
            best_left = max(best_left, values[j] + j)
        return ans


print([Solution().maxScoreSightseeingPair(v) for v, _ in examples])
assert all(Solution().maxScoreSightseeingPair(v) == w for v, w in examples)

[11, 2, 9, 17]


## 10. Vì sao đúng

**(a) Tách biến.** Với `i < j`: `values[i] + values[j] + i − j = (values[i] + i) + (values[j] − j) = A[i] + B[j]`. Đáp án `= max_{i<j} (A[i] + B[j])`.

**(b) Cực đại tiền tố.** Với `j` cố định, `max_{i<j} (A[i] + B[j]) = B[j] + max_{i<j} A[i]` (`B[j]` không phụ thuộc `i`). Vậy đáp án `= max_{j ≥ 1} (B[j] + max_{i<j} A[i])`.

**(c) Một lượt.** Bất biến: trước khi xử lý `j`, `best_left = max_{i<j} A[i]`. Ta tính điểm của `j` bằng `best_left + B[j]` rồi cập nhật `best_left = max(best_left, A[j])`, khôi phục bất biến cho `j + 1`. Cập nhật **sau** tính điểm đảm bảo `i < j` (không có `i = j`).

**(d) Cách 3** là cùng công thức với `pm[j−1] = max_{i<j} A[i]`.

**(e) Chia để trị.** Cặp `(i, j)` hoặc nằm cùng một nửa (đệ quy) hoặc `i` ở nửa trái, `j` ở nửa phải. Trong trường hợp sau `i < j` tự động đúng và điểm tốt nhất là `max A (trái) + max B (phải)`. Đáy `l = r`: không có cặp (`best = −∞`) nhưng cung cấp `A`, `B`.

**(f) Quét ngược.** Đối xứng: với `i` cố định, `max_{j>i} (A[i] + B[j]) = A[i] + max_{j>i} B[j]`; giữ `best_right = max_{j>i} B[j]` và cập nhật sau.

**(g) Tính chất kiểm tra được.** Đảo mảng không đổi đáp án (cặp `(i, j)` ↔ `(n−1−j, n−1−i)` cùng điểm); cộng hằng `c` vào mọi giá trị tăng đáp án thêm `2c`; đáp án `≥ max_i (values[i] + values[i+1] − 1)` (cặp kề); đáp án `≤ 2·max(values) − 1`.

**(h) Kiểm chứng bằng máy.** Mọi cách khớp oracle brute `O(n²)` trên **mọi** mảng độ dài `2..6` với giá trị trong `[1, 4]` và độ dài `2..4` trong `[1, 8]`; khớp brute trên mảng ngẫu nhiên tới `n = 3000`; các cách `O(n)` khớp nhau tới `n = 5·10^4` (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử mọi cặp | `O(n²)` | `O(1)` | oracle |
| 2. Một lượt `best_left` | `O(n)` | `O(1)` | **đáp án chính** |
| 3. Hai mảng cực đại tiền tố | `O(n)` | `O(n)` | dạng khai báo |
| 4. Chia để trị | `O(n)` | `O(log n)` | khung cho biến thể |
| 5. Quét ngược | `O(n)` | `O(1)` | đối xứng |

Đo thực tế `n = 5·10^4`:

In [17]:
def bench(fn, arg, repeat=5):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(arg)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(1014)
big = [rng.randint(1, 1000) for _ in range(50000)]
for name, fn in [("one_pass", sight_one_pass), ("prefix_arrays", sight_prefix_arrays), ("divide", sight_divide), ("scan_right", sight_scan_right)]:
    print(f"{name:14s}: {bench(fn, big) * 1e3:7.2f} ms (n = 5e4) -> {fn(big)}")
print(f"{'brute':14s}: {bench(sight_brute, big[:1500], repeat=1) * 1e3:7.1f} ms (n = 1500)")

one_pass      :   10.62 ms (n = 5e4) -> 1997
prefix_arrays :    9.58 ms (n = 5e4) -> 1997


divide        :   24.74 ms (n = 5e4) -> 1997
scan_right    :    5.76 ms (n = 5e4) -> 1997


brute         :    96.2 ms (n = 1500)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi mảng nhỏ, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "one_pass": sight_one_pass,
    "prefix_arrays": sight_prefix_arrays,
    "divide": sight_divide,
    "scan_right": sight_scan_right,
    "brute": sight_brute,
    "Solution": lambda v: Solution().maxScoreSightseeingPair(list(v)),
    "SolutionOnePass": lambda v: SolutionOnePass().maxScoreSightseeingPair(list(v)),
    "SolutionPrefixArrays": lambda v: SolutionPrefixArrays().maxScoreSightseeingPair(list(v)),
    "SolutionDivide": lambda v: SolutionDivide().maxScoreSightseeingPair(list(v)),
    "SolutionScanRight": lambda v: SolutionScanRight().maxScoreSightseeingPair(list(v)),
    "SolutionBrute": lambda v: SolutionBrute().maxScoreSightseeingPair(list(v)),
}
SLOW = {"brute", "SolutionBrute"}
fast = [n for n in IMPLS if n not in SLOW]


def check(v, want=None, slow_limit=3000):
    if want is None and len(v) <= slow_limit:
        want = sight_brute(v)
    results = {}
    for name in IMPLS:
        if name in SLOW and len(v) > slow_limit:
            continue
        results[name] = IMPLS[name](v)
    assert len(set(results.values())) == 1, (v[:8], results)
    got = next(iter(results.values()))
    if want is not None:
        assert got == want, (v[:8], got, want)
    return got


for v, want in examples:
    check(v, want)
print("ví dụ đề: oracle brute và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle brute và 11 cài đặt đúng


### B. Edge case

- `n = 2`: `[1, 2]` → `2`; `[1000, 1000]` → `1999`; `[1, 1]` → `1`.
- Toàn bằng nhau: điểm `2v − 1` (cặp kề).
- Tăng/giảm dần.
- Giá trị lớn cách xa nhau (khoảng cách phạt nặng): `[1000, 1, 1, 1, 1000]` → `1000 + 1000 − 4 = 1996`? (so với kề `1000 + 1 − 1 = 1000`).
- Giá trị nhỏ cách xa: `[1, 1000, 1, 1, 1]`.
- Cặp tốt nhất ở cuối / ở đầu / ở giữa.

In [19]:
edge = [
    [1, 2], [2, 1], [1000, 1000], [1, 1], [5, 5, 5], [5, 5, 5, 5, 5], [1, 2, 3, 4, 5], [5, 4, 3, 2, 1], [1000, 1, 1, 1, 1000], [1, 1000, 1, 1, 1],
    [1, 1, 1, 1000, 1000], [1000, 1000, 1, 1, 1], [3, 7, 2, 3], [10, 1, 1, 10], [1, 3, 5], [8, 1, 5, 2, 6], [1, 1000] + [1] * 20 + [1000],
]
for v in edge:
    check(v)
print(len(edge), "edge case: mọi cài đặt khớp brute")
print("[1000,1,1,1,1000] ->", Solution().maxScoreSightseeingPair([1000, 1, 1, 1, 1000]), "| [1000,1000] ->", Solution().maxScoreSightseeingPair([1000, 1000]))

17 edge case: mọi cài đặt khớp brute
[1000,1,1,1,1000] -> 1996 | [1000,1000] -> 1999


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_update_first(values):                           # cập nhật best_left trước khi tính điểm (cho phép i = j)
    best_left, ans = values[0], float("-inf")
    for j in range(1, len(values)):
        best_left = max(best_left, values[j] + j)
        ans = max(ans, best_left + values[j] - j)
    return ans


def bug_wrong_sign(values):                             # dùng values[i] - i và values[j] + j
    best_left, ans = values[0], float("-inf")
    for j in range(1, len(values)):
        ans = max(ans, best_left + values[j] + j)
        best_left = max(best_left, values[j] - j)
    return ans


def bug_adjacent_only(values):                          # chỉ xét cặp kề nhau
    return max(values[i] + values[i + 1] - 1 for i in range(len(values) - 1))


def bug_two_largest(values):                            # chọn hai giá trị lớn nhất rồi trừ khoảng cách
    idx = sorted(range(len(values)), key=lambda i: -values[i])[:2]
    i, j = sorted(idx)
    return values[i] + values[j] + i - j


def bug_no_decay(values):                               # best_left = max values[i] (quên + i)
    best, ans = values[0], float("-inf")
    for j in range(1, len(values)):
        ans = max(ans, best + values[j] - j)
        best = max(best, values[j])
    return ans


def bug_index_sum(values):                              # values[i] + values[j] + i + j
    n = len(values)
    return max(values[i] + values[j] + i + j for i in range(n) for j in range(i + 1, n))


def bug_init_zero(values):                              # best_left khởi tạo 0 thay vì values[0]
    best_left, ans = 0, float("-inf")
    for j in range(1, len(values)):
        ans = max(ans, best_left + values[j] - j)
        best_left = max(best_left, values[j] + j)
    return ans


BUGS = [bug_update_first, bug_wrong_sign, bug_adjacent_only, bug_two_largest, bug_no_decay, bug_index_sum, bug_init_zero]
pool = [list(a) for n in range(2, 6) for a in itertools.product(range(1, 6), repeat=n)]
for bug in BUGS:
    found = next((a for a in pool if bug(a) != sight_brute(a)), None)
    assert found is not None, bug.__name__
    print(f"{bug.__name__:20s} sai trên {found}: ra {bug(found)}, đúng {sight_brute(found)}")

bug_update_first     sai trên [1, 1]: ra 2, đúng 1
bug_wrong_sign       sai trên [1, 1]: ra 3, đúng 1
bug_adjacent_only    sai trên [3, 1, 3]: ra 3, đúng 4
bug_two_largest      sai trên [1, 1, 2]: ra 1, đúng 2
bug_no_decay         sai trên [1, 1, 2]: ra 1, đúng 2
bug_index_sum        sai trên [1, 1]: ra 3, đúng 1
bug_init_zero        sai trên [1, 1]: ra 0, đúng 1


### D. Vét cạn mọi mảng nhỏ

Mọi mảng độ dài `2..6` với giá trị trong `[1, 4]` (`16 + 64 + 256 + 1024 + 4096 = 5 456` mảng) và độ dài `2..4` với giá trị trong `[1, 8]` (`64 + 512 + 4096 = 4 672` mảng). Oracle `sight_brute`.

In [21]:
t0 = time.perf_counter()
cnt = 0
for hi_val, max_n in ((4, 6), (8, 4)):
    for n in range(2, max_n + 1):
        for arr in itertools.product(range(1, hi_val + 1), repeat=n):
            a = list(arr)
            want = sight_brute(a)
            for name in IMPLS:
                assert IMPLS[name](a) == want, (name, a, IMPLS[name](a), want)
            cnt += 1
print(cnt, "mảng (độ dài 2..6 trong [1,4]; 2..4 trong [1,8]):", len(IMPLS), "cài đặt khớp oracle brute;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 5456 + 4672

10128 mảng (độ dài 2..6 trong [1,4]; 2..4 trong [1,8]): 11 cài đặt khớp oracle brute; 0.2 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Khớp brute (`n ≤ 3000`); (2) đảo mảng không đổi đáp án; (3) cộng hằng `c` vào mọi giá trị tăng đáp án `2c`; (4) `đáp án ≥ max(values[i] + values[i+1] − 1)` (cặp kề); (5) `đáp án ≤ 2·max − 1`; (6) đáp án hợp lệ: tồn tại cặp `(i, j)` đạt đúng điểm đó (dựng bằng truy vết); (7) thêm một phần tử ở cuối không làm đáp án giảm.

In [22]:
def best_pair(values):
    """Truy vết một cặp (i, j) đạt điểm tối đa."""
    best_left, best_i = values[0], 0
    ans, pair = float("-inf"), None
    for j in range(1, len(values)):
        s = best_left + values[j] - j
        if s > ans:
            ans, pair = s, (best_i, j)
        if values[j] + j > best_left:
            best_left, best_i = values[j] + j, j
    return ans, pair


rng = random.Random(20240112)
t0 = time.perf_counter()
for it in range(500):
    n = rng.choice([2, 3, 5, 12, 50, 300, 3000])
    span = rng.choice([1, 3, 10, 1000])
    v = [rng.randint(1, span) for _ in range(n)]
    want = sight_brute(v)                                                                    # (1)
    for name in fast:
        assert IMPLS[name](v) == want, (name, it)
    assert Solution().maxScoreSightseeingPair(v[::-1]) == want                               # (2)
    c = rng.randint(1, 100)
    assert Solution().maxScoreSightseeingPair([x + c for x in v]) == want + 2 * c            # (3)
    assert want >= max(v[i] + v[i + 1] - 1 for i in range(n - 1))                            # (4)
    assert want <= 2 * max(v) - 1                                                            # (5)
    a, (i, j) = best_pair(v)
    assert a == want and i < j and v[i] + v[j] + i - j == want                               # (6)
    assert Solution().maxScoreSightseeingPair(v + [rng.randint(1, span)]) >= want            # (7)
print("500 mảng ngẫu nhiên (n tới 3000): 7 tính chất đúng;", round(time.perf_counter() - t0, 1), "s")

500 mảng ngẫu nhiên (n tới 3000): 7 tính chất đúng; 27.0 s


### F. Giới hạn

`n = 5·10^4`: toàn giá trị lớn nhất `1000`, toàn `1`, tăng dần, giảm dần, ngẫu nhiên, đỉnh nhọn ở hai đầu; các cách `O(n)` khớp nhau.

In [23]:
t0 = time.perf_counter()
n = 50000
rng = random.Random(10141)
cases = [
    ([1000] * n, 1999), ([1] * n, 1), ([min(1000, 1 + i // 50) for i in range(n)], None), ([max(1, 1000 - i // 50) for i in range(n)], None),
    ([rng.randint(1, 1000) for _ in range(n)], None), ([1000] + [1] * (n - 2) + [1000], None), ([1] + [1000] * (n - 1), None),
]
for v, want in cases:
    res = {name: IMPLS[name](v) for name in fast}
    assert len(set(res.values())) == 1, res
    if want is not None:
        assert res["Solution"] == want
    print(f"  đáp án = {res['Solution']}")
assert Solution().maxScoreSightseeingPair([1000] + [1] * (n - 2) + [1000]) == 1000      # khoảng cách n-1 quá lớn: cặp tốt nhất là cặp kề (1000 + 1 - 1)
print("n = 5·10^4: các cài đặt nhanh khớp nhau;", round(time.perf_counter() - t0, 2), "s")

  đáp án = 1999


  đáp án = 1
  đáp án = 1999


  đáp án = 1999


  đáp án = 1995
  đáp án = 1000


  đáp án = 1999
n = 5·10^4: các cài đặt nhanh khớp nhau; 0.68 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: cập nhật `best_left` trước khi tính điểm

In [24]:
v = [8, 1, 5, 2, 6]
print("bug:", bug_update_first(v), "| đúng:", Solution().maxScoreSightseeingPair(v))
assert bug_update_first(v) == 12 and Solution().maxScoreSightseeingPair(v) == 11

bug: 12 | đúng: 11


**Vì sao sai:** nếu `best_left` đã chứa `values[j] + j` thì ta ghép `j` với chính nó: điểm `2·values[j]` (không hợp lệ vì `i < j`). **Đúng:** tính điểm trước, cập nhật sau.

### Lỗi 2: nhầm dấu khoảng cách

In [25]:
v = [8, 1, 5, 2, 6]
print("bug:", bug_wrong_sign(v), "| bug (i+j):", bug_index_sum(v), "| đúng:", Solution().maxScoreSightseeingPair(v))
assert bug_wrong_sign(v) != 11 and bug_index_sum(v) != 11

bug: 18 | bug (i+j): 18 | đúng: 11


**Vì sao sai:** khoảng cách **bị trừ**: `... + i − j` với `i < j`, nên cặp càng xa càng lỗ. Phần theo `i` là `values[i] + i` (không `− i`), phần theo `j` là `values[j] − j`.

### Lỗi 3: chỉ xét cặp kề nhau

In [26]:
v = [1000, 1, 1, 1, 1000]
print("bug:", bug_adjacent_only(v), "| đúng:", Solution().maxScoreSightseeingPair(v))
assert bug_adjacent_only(v) < Solution().maxScoreSightseeingPair(v)

bug: 1000 | đúng: 1996


**Vì sao sai:** hai điểm rất hay có thể bù được khoảng cách xa: `1000 + 1000 − 4 = 1996` lớn hơn mọi cặp kề.

### Lỗi 4: chọn hai giá trị lớn nhất rồi trừ khoảng cách

In [27]:
bad = next(a for a in pool if bug_two_largest(a) != sight_brute(a))
print("phản ví dụ nhỏ:", bad, "| bug:", bug_two_largest(bad), "| đúng:", sight_brute(bad))
assert bug_two_largest(bad) < sight_brute(bad)

phản ví dụ nhỏ: [1, 1, 2] | bug: 1 | đúng: 2


**Vì sao sai:** hai giá trị lớn nhất có thể ở xa nhau (phạt lớn) trong khi một cặp kề cho điểm tốt hơn. Phải cân bằng giá trị và khoảng cách bằng công thức tách biến.

### Lỗi 5: quên cộng chỉ số (`best_left = max values[i]`)

In [28]:
v = [8, 1, 5, 2, 6]
print("bug:", bug_no_decay(v), "| đúng:", Solution().maxScoreSightseeingPair(v))
bad = next(a for a in pool if bug_no_decay(a) != sight_brute(a))
print("phản ví dụ nhỏ:", bad, "| bug:", bug_no_decay(bad), "| đúng:", sight_brute(bad))
assert bug_no_decay(bad) != sight_brute(bad)

bug: 11 | đúng: 11
phản ví dụ nhỏ: [1, 1, 2] | bug: 1 | đúng: 2


**Vì sao sai:** `best_left` phải là `max(values[i] + i)`, vì phần theo `i` của điểm là `values[i] + i`; bỏ `+ i` làm sai thứ hạng các `i`.

### Lỗi 6 (bonus): khởi tạo `best_left = 0`

In [29]:
v = [1, 2]
print("bug:", bug_init_zero(v), "| đúng:", Solution().maxScoreSightseeingPair(v))
assert bug_init_zero(v) != 2

bug: 1 | đúng: 2


**Vì sao sai:** `best_left` phải bắt đầu bằng `values[0] + 0` (cặp `(0, j)` phải khả dụng); `0` thấp hơn nên bỏ sót.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "tối đa hoá `f(i) + g(j)` với `i < j`" (hàm của `i` cộng hàm của `j`) → **tách biến** và duy trì cực đại tiền tố của `f`.

**Mẹo:**

1. Viết điểm dạng `A[i] + B[j]` bằng cách dồn các số hạng theo chỉ số.
2. Duyệt `j`, giữ `max A[i]` cho `i < j`; **tính điểm trước, cập nhật sau**.
3. `accumulate(A, max)` cho cực đại tiền tố.
4. Với ràng buộc cửa sổ `j − i ≤ w`: hàng đợi đơn điệu (xem mục 15).
5. Oracle `O(n²)` theo định nghĩa.
6. Kiểm tra tính chất: đảo mảng, cộng hằng.

**Họ bài cùng pattern:** Best Time to Buy and Sell Stock (121), Maximum Difference Between Increasing Elements (2016), Maximum Subarray (53), Maximum Score of a Good Subarray (1793), Sliding Window Maximum (239), Constrained Subsequence Sum (1425).

## 15. Biến thể

1. **Hệ số phạt khoảng cách `λ`:** `values[i] + values[j] − λ·(j − i)`.
2. **Giới hạn khoảng cách `j − i ≤ w`:** dùng **hàng đợi đơn điệu** cho `max A[i]` trong cửa sổ.
3. **Trả về cặp chỉ số** (đã làm ở mục 12E).

Cài đặt và đối chiếu brute:

In [30]:
def sight_lambda(values, lam):
    best_left = values[0]                              # max(values[i] + lam*i)
    ans = float("-inf")
    for j in range(1, len(values)):
        ans = max(ans, best_left + values[j] - lam * j)
        best_left = max(best_left, values[j] + lam * j)
    return ans


def sight_window(values, w):
    """Cặp (i, j) với 1 <= j - i <= w: hàng đợi đơn điệu giảm của A[i] = values[i] + i."""
    dq = deque([0])                                    # chỉ số i, A giảm dần
    ans = float("-inf")
    for j in range(1, len(values)):
        while dq and j - dq[0] > w:
            dq.popleft()                               # i quá xa
        if dq:
            ans = max(ans, values[dq[0]] + dq[0] + values[j] - j)
        a = values[j] + j
        while dq and values[dq[-1]] + dq[-1] <= a:
            dq.pop()
        dq.append(j)
    return ans


def brute_lambda(values, lam):
    n = len(values)
    return max(values[i] + values[j] - lam * (j - i) for i in range(n) for j in range(i + 1, n))


def brute_window(values, w):
    n = len(values)
    return max(values[i] + values[j] + i - j for i in range(n) for j in range(i + 1, min(n, i + w + 1)))


rng = random.Random(15)
for _ in range(500):
    n = rng.randint(2, 14)
    v = [rng.randint(1, 20) for _ in range(n)]
    lam = rng.choice([0, 1, 2, 5])
    w = rng.randint(1, n)
    assert sight_lambda(v, lam) == brute_lambda(v, lam)                  # (1)
    assert sight_window(v, w) == brute_window(v, w)                      # (2)
assert sight_lambda([8, 1, 5, 2, 6], 1) == 11 and sight_window([8, 1, 5, 2, 6], 10) == 11
print("hệ số phạt λ và cửa sổ khoảng cách: khớp brute")

hệ số phạt λ và cửa sổ khoảng cách: khớp brute


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi cặp | định nghĩa | `O(n²)` | `O(1)` | oracle |
| 2. Một lượt `best_left` | tách biến | `O(n)` | `O(1)` | **đáp án chính** |
| 3. Hai mảng cực đại tiền tố | `accumulate` | `O(n)` | `O(n)` | dạng khai báo |
| 4. Chia để trị | `maxA(trái) + maxB(phải)` | `O(n)` | `O(log n)` | nền cho bài phức tạp hơn |
| 5. Quét ngược | cực đại hậu tố của `B` | `O(n)` | `O(1)` | đối xứng |

Mọi cách khớp oracle brute trên toàn bộ mảng nhỏ (mục 12D) và khớp nhau tới `n = 5·10^4`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [31]:
print(Solution().maxScoreSightseeingPair([8, 1, 5, 2, 6]), Solution().maxScoreSightseeingPair([1, 2]), Solution().maxScoreSightseeingPair([5, 5, 5]),
      Solution().maxScoreSightseeingPair([10, 1, 1, 10]), Solution().maxScoreSightseeingPair([1, 3, 5]))

11 2 9 17 7


### Bài 2 (tính tay)

`values = [3, 7, 2, 3]`. (a) Tính `A` và `B`. (b) Với mỗi `j = 1, 2, 3` tìm `best_left` và điểm. (c) Đáp án và cặp? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện cách một lượt:

In [32]:
def one_pass_todo(values):
    best_left = values[0]
    ans = float("-inf")
    for j in range(1, len(values)):
        pass            # TODO 1: ans = max(ans, best_left + values[j] - j)
        pass            # TODO 2: best_left = max(best_left, values[j] + j)
    return ans


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(one_pass_todo(v) == w for v, w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **quét ngược** (`best_right = max(values[j] − j)`):

In [33]:
class MySolution:
    def maxScoreSightseeingPair(self, values):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().maxScoreSightseeingPair(list(v)) == w for v, w in examples)

### Bài 5 (mini challenge): giới hạn khoảng cách `w`

Viết `with_window(values, w)` trả về điểm lớn nhất của cặp `(i, j)` với `1 ≤ j − i ≤ w`. Gợi ý: hàng đợi đơn điệu giảm theo `values[i] + i`. Kiểm bằng brute.

In [34]:
def with_window(values, w):
    pass            # TODO


# Khi làm xong:
# assert with_window([8, 1, 5, 2, 6], 1) == 8 and with_window([8, 1, 5, 2, 6], 2) == 11

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `score = A[i] + B[j]`.
- Mức 2: `[10,1,1,10]`: `10 + 10 + 0 − 3 = 17`; `[1,3,5]`: cặp `(1, 2)`: `3 + 5 − 1 = 7`.
- Mức 3: `11 2 9 17 7`.

**Bài 2**

- Mức 1: `A = [3, 8, 4, 6]`, `B = [3, 6, 0, 0]`.
- Mức 2: `j = 1`: `best_left = 3` ⇒ `3 + 6 = 9`; `j = 2`: `best_left = 8` ⇒ `8 + 0 = 8`; `j = 3`: `best_left = 8` ⇒ `8 + 0 = 8`.
- Mức 3: đáp án `9` tại `(0, 1)`.

**Bài 3**

- Mức 1: tính điểm trước.
- Mức 2: `ans = max(ans, best_left + values[j] - j)`.
- Mức 3: sau đó `best_left = max(best_left, values[j] + j)`.

**Bài 4**

- Mức 1: `best_right = values[n-1] - (n-1)`.
- Mức 2: `for i in range(n-2, -1, -1): ans = max(ans, values[i] + i + best_right)`.
- Mức 3: cập nhật `best_right = max(best_right, values[i] - i)` sau.

**Bài 5**

- Mức 1: giữ deque các chỉ số `i` với `A[i]` giảm dần.
- Mức 2: bỏ đầu deque khi `j - i > w`.
- Mức 3: điểm với `j` là `A[deque[0]] + B[j]`; trước khi thêm `j`, pop cuối khi `A[cuối] <= A[j]`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [35]:
vals = (Solution().maxScoreSightseeingPair([8, 1, 5, 2, 6]), Solution().maxScoreSightseeingPair([1, 2]), Solution().maxScoreSightseeingPair([5, 5, 5]),
        Solution().maxScoreSightseeingPair([10, 1, 1, 10]), Solution().maxScoreSightseeingPair([1, 3, 5]))
print(vals)
assert vals == (11, 2, 9, 17, 7)
print("Bài 1 OK")

(11, 2, 9, 17, 7)
Bài 1 OK


### Bài 2

In [36]:
v = [3, 7, 2, 3]
A = [x + i for i, x in enumerate(v)]
B = [x - j for j, x in enumerate(v)]
print("A =", A, "| B =", B)
assert A == [3, 8, 4, 6] and B == [3, 6, 0, 0]
ans, pair = best_pair(v)
assert ans == 9 == sight_brute(v) and pair == (0, 1)
print("Bài 2 OK: đáp án 9 tại", pair)

A = [3, 8, 4, 6] | B = [3, 6, 0, 0]
Bài 2 OK: đáp án 9 tại (0, 1)


### Bài 3

In [37]:
def one_pass_answer(values):
    best_left = values[0]
    ans = float("-inf")
    for j in range(1, len(values)):
        ans = max(ans, best_left + values[j] - j)
        best_left = max(best_left, values[j] + j)
    return ans


assert all(one_pass_answer(v) == w for v, w in examples)
rng = random.Random(3)
for _ in range(300):
    v = [rng.randint(1, 30) for _ in range(rng.randint(2, 20))]
    assert one_pass_answer(v) == sight_brute(v)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [38]:
class MySolutionAnswer:
    def maxScoreSightseeingPair(self, values):
        n = len(values)
        best_right = values[n - 1] - (n - 1)
        ans = float("-inf")
        for i in range(n - 2, -1, -1):
            ans = max(ans, values[i] + i + best_right)
            best_right = max(best_right, values[i] - i)
        return ans


assert all(MySolutionAnswer().maxScoreSightseeingPair(list(v)) == w for v, w in examples)
rng = random.Random(4)
for _ in range(300):
    v = [rng.randint(1, 30) for _ in range(rng.randint(2, 20))]
    assert MySolutionAnswer().maxScoreSightseeingPair(v) == sight_brute(v)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [39]:
def with_window(values, w):
    dq = deque([0])
    ans = float("-inf")
    for j in range(1, len(values)):
        while dq and j - dq[0] > w:
            dq.popleft()
        if dq:
            ans = max(ans, values[dq[0]] + dq[0] + values[j] - j)
        a = values[j] + j
        while dq and values[dq[-1]] + dq[-1] <= a:
            dq.pop()
        dq.append(j)
    return ans


assert with_window([8, 1, 5, 2, 6], 1) == 8 and with_window([8, 1, 5, 2, 6], 2) == 11
rng = random.Random(5)
for _ in range(300):
    v = [rng.randint(1, 30) for _ in range(rng.randint(2, 20))]
    w = rng.randint(1, len(v))
    assert with_window(v, w) == brute_window(v, w)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (tối đa hoá `f(i) + g(j)` với `i < j` → tách biến + cực đại tiền tố):**

```python
class Solution:
    def maxScoreSightseeingPair(self, values: List[int]) -> int:
        best_left = values[0]
        ans = float("-inf")
        for j in range(1, len(values)):
            ans = max(ans, best_left + values[j] - j)
            best_left = max(best_left, values[j] + j)
        return ans
```

**Checklist trước khi submit:**

- [ ] Phần theo `i` là `values[i] + i`, phần theo `j` là `values[j] − j`?
- [ ] Tính điểm **trước**, cập nhật `best_left` **sau** (để `i < j`)?
- [ ] `best_left` khởi tạo `values[0] + 0`; vòng `j` bắt đầu từ `1`?
- [ ] Đã thử `[8,1,5,2,6] → 11`, `[1,2] → 2`, `[10,1,1,10] → 17`, toàn bằng nhau?
- [ ] Không chỉ xét cặp kề nhau / hai giá trị lớn nhất?
- [ ] Hiểu tính chất đối xứng khi đảo mảng và cộng hằng (`+2c`)?
- [ ] Đúng tên `class Solution`, `maxScoreSightseeingPair(self, values)`, không in debug.